# Session 11 · Data Handling Using Pandas

Run each cell with **Shift+Enter**. Original material by Aseef Ahmed.

In [1]:
# Setup: run from the code/ folder so local modules and data files are found
import os
if os.path.isdir('code'):
    os.chdir('code')

## Example 1 - create and inspect

In [2]:
from pathlib import Path
import pandas as pd
OUT = Path("output"); OUT.mkdir(exist_ok=True)

s = pd.Series([72, 91, 64], index=["Ali", "Sana", "Omar"], name="marks")
print(s, "\nmean:", s.mean())

df = pd.DataFrame({
    "name": ["Ayesha", "Bilal", "Hira", "Omar", "Zara", "Hamza"],
    "city": ["Karachi", "Lahore", "Karachi", "Quetta", "Lahore", "Karachi"],
    "python": [88, 64, 92, None, 75, 58],
    "sql": [79, 70, 95, 60, None, 66],
})
print(df.shape)
print(df.dtypes)
print(df.describe().round(1))

Ali     72
Sana    91
Omar    64
Name: marks, dtype: int64 
mean: 75.66666666666667
(6, 4)
name          str
city          str
python    float64
sql       float64
dtype: object
       python   sql
count     5.0   5.0
mean     75.4  74.0
std      14.7  13.6
min      58.0  60.0
25%      64.0  66.0
50%      75.0  70.0
75%      88.0  79.0
max      92.0  95.0


## Example 2 - select and filter

In [3]:
print(df.loc[df["city"] == "Karachi", ["name", "python"]])
print(df.iloc[:2])
print(df.query("python >= 75 and sql >= 75"))
print(df[df["city"].isin(["Lahore", "Quetta"])]["name"].tolist())

     name  python
0  Ayesha    88.0
2    Hira    92.0
5   Hamza    58.0
     name     city  python   sql
0  Ayesha  Karachi    88.0  79.0
1   Bilal   Lahore    64.0  70.0
     name     city  python   sql
0  Ayesha  Karachi    88.0  79.0
2    Hira  Karachi    92.0  95.0
['Bilal', 'Omar', 'Zara']


## Example 3 - handle missing values

In [4]:
print(df.isna().sum())
df["python"] = df["python"].fillna(df["python"].mean())
df = df.dropna(subset=["sql"]).reset_index(drop=True)
print(df)

name      0
city      0
python    1
sql       1
dtype: int64
     name     city  python   sql
0  Ayesha  Karachi    88.0  79.0
1   Bilal   Lahore    64.0  70.0
2    Hira  Karachi    92.0  95.0
3    Omar   Quetta    75.4  60.0
4   Hamza  Karachi    58.0  66.0


## Example 4 - new columns, apply and sort

In [5]:
df["total"] = df["python"] + df["sql"]
df["grade"] = df["total"].apply(lambda t: "A" if t >= 160 else "B" if t >= 130 else "C")
print(df.sort_values("total", ascending=False))

     name     city  python   sql  total grade
2    Hira  Karachi    92.0  95.0  187.0     A
0  Ayesha  Karachi    88.0  79.0  167.0     A
3    Omar   Quetta    75.4  60.0  135.4     B
1   Bilal   Lahore    64.0  70.0  134.0     B
4   Hamza  Karachi    58.0  66.0  124.0     C


## Example 5 - groupby and pivot tables

In [6]:
print(df.groupby("city")["total"].agg(["count", "mean", "max"]).round(1))
print(pd.pivot_table(df, values="total", index="city", columns="grade", aggfunc="count", fill_value=0))

         count   mean    max
city                        
Karachi      3  159.3  187.0
Lahore       1  134.0  134.0
Quetta       1  135.4  135.4
grade    A  B  C
city            
Karachi  2  0  1
Lahore   0  1  0
Quetta   0  1  0


## Example 6 - merge two tables

In [7]:
fees = pd.DataFrame({"name": ["Ayesha", "Hira", "Omar"], "fee_paid": [True, True, False]})
print(df.merge(fees, on="name", how="left"))

     name     city  python   sql  total grade fee_paid
0  Ayesha  Karachi    88.0  79.0  167.0     A     True
1   Bilal   Lahore    64.0  70.0  134.0     B      NaN
2    Hira  Karachi    92.0  95.0  187.0     A     True
3    Omar   Quetta    75.4  60.0  135.4     B    False
4   Hamza  Karachi    58.0  66.0  124.0     C      NaN


## Example 7 - save and reload

In [8]:
df.to_csv(OUT / "students.csv", index=False)
back = pd.read_csv(OUT / "students.csv")
print(back.equals(df), back.columns.tolist())

True ['name', 'city', 'python', 'sql', 'total', 'grade']
